In [3]:
# Project 4: Diabetes Population Health Management
# Notebook 6: Geographic Analysis and Mapping
# Objective: Identify state-level hotspots for intervention targeting

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.weightstats import DescrStatsW
import warnings
import os
import json
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Environment loaded successfully")

# Load processed data
df = pd.read_csv('../data/processed/brfss_2023_diabetes.csv')

print(f"Loaded: {len(df):,} respondents")
print(f"States represented: {df['_STATE'].nunique()}")

# Weighted prevalence function (from Notebook 2)
def weighted_prevalence(data, outcome_var, weight_var='_LLCPWT'):
    """Calculate weighted prevalence with 95% confidence interval"""
    valid = data[[outcome_var, weight_var]].dropna()
    
    if len(valid) == 0 or len(valid) < 30:  # Minimum sample size
        return {
            'prevalence': np.nan,
            'ci_lower': np.nan,
            'ci_upper': np.nan,
            'n': len(valid)
        }
    
    weighted_stats = DescrStatsW(valid[outcome_var], weights=valid[weight_var])
    prevalence = weighted_stats.mean
    ci_lower, ci_upper = weighted_stats.tconfint_mean()
    
    return {
        'prevalence': prevalence,
        'ci_lower': max(0, ci_lower),
        'ci_upper': min(1, ci_upper),
        'n': len(valid)
    }

print("\nWeighted prevalence function loaded")

# Calculate state-level prevalence
print("\nCalculating state-level diabetes prevalence...")

state_results = []
for state_code in sorted(df['_STATE'].dropna().unique()):
    subset = df[df['_STATE'] == state_code]
    result = weighted_prevalence(subset, 'has_diabetes')
    
    if not np.isnan(result['prevalence']):
        state_results.append({
            'state_code': int(state_code),
            'prevalence': result['prevalence'],
            'ci_lower': result['ci_lower'],
            'ci_upper': result['ci_upper'],
            'n': result['n'],
            'diabetes_cases': int(subset['has_diabetes'].sum())
        })

state_df = pd.DataFrame(state_results)
state_df = state_df.sort_values('prevalence', ascending=False)

print(f"States with sufficient data: {len(state_df)}")

# State code to name mapping (FIPS codes)
state_names = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California',
    8: 'Colorado', 9: 'Connecticut', 10: 'Delaware', 11: 'DC', 12: 'Florida',
    13: 'Georgia', 15: 'Hawaii', 16: 'Idaho', 17: 'Illinois', 18: 'Indiana',
    19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 22: 'Louisiana', 23: 'Maine',
    24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan', 27: 'Minnesota',
    28: 'Mississippi', 29: 'Missouri', 30: 'Montana', 31: 'Nebraska',
    32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey', 35: 'New Mexico',
    36: 'New York', 37: 'North Carolina', 38: 'North Dakota', 39: 'Ohio',
    40: 'Oklahoma', 41: 'Oregon', 42: 'Pennsylvania', 44: 'Rhode Island',
    45: 'South Carolina', 46: 'South Dakota', 47: 'Tennessee', 48: 'Texas',
    49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington',
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}

state_df['state_name'] = state_df['state_code'].map(state_names)

print("\nTop 10 States by Diabetes Prevalence:")
print("="*70)
for _, row in state_df.head(10).iterrows():
    print(f"{row['state_name']:<20} {row['prevalence']:>6.1%}  [{row['ci_lower']:.1%}, {row['ci_upper']:.1%}]  n={row['n']:,}")
print("="*70)

print("\nBottom 10 States by Diabetes Prevalence:")
print("="*70)
for _, row in state_df.tail(10).iterrows():
    print(f"{row['state_name']:<20} {row['prevalence']:>6.1%}  [{row['ci_lower']:.1%}, {row['ci_upper']:.1%}]  n={row['n']:,}")
print("="*70)

# Geographic disparity analysis
prevalence_range = state_df['prevalence'].max() - state_df['prevalence'].min()
mean_prevalence = state_df['prevalence'].mean()
std_prevalence = state_df['prevalence'].std()

print(f"\nGeographic Disparity Summary:")
print(f"  Mean state prevalence: {mean_prevalence:.2%}")
print(f"  Standard deviation: {std_prevalence:.2%}")
print(f"  Range: {prevalence_range:.2%} ({state_df['prevalence'].min():.1%} to {state_df['prevalence'].max():.1%})")
print(f"  Highest state: {state_df.iloc[0]['state_name']} ({state_df.iloc[0]['prevalence']:.1%})")
print(f"  Lowest state: {state_df.iloc[-1]['state_name']} ({state_df.iloc[-1]['prevalence']:.1%})")

# Identify high-burden states (top quartile)
high_burden_threshold = state_df['prevalence'].quantile(0.75)
high_burden_states = state_df[state_df['prevalence'] >= high_burden_threshold]

print(f"\nHigh-Burden States (Top 25%):")
print(f"  Threshold: >{high_burden_threshold:.1%}")
print(f"  Number of states: {len(high_burden_states)}")
valid_names = [str(s) for s in high_burden_states['state_name'].tolist() if pd.notna(s)]
print(f"  States: {', '.join(valid_names) if valid_names else 'See state codes'}")

# Create outputs
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

# Figure 1: State prevalence bar chart (top 20)
fig, ax = plt.subplots(figsize=(12, 10))

top_20 = state_df.head(20)
y_pos = np.arange(len(top_20))
colors = ['red' if x >= high_burden_threshold else 'steelblue' for x in top_20['prevalence']]

ax.barh(y_pos, top_20['prevalence'].values, color=colors, alpha=0.7)

for i in range(len(top_20)):
    ax.errorbar(top_20.iloc[i]['prevalence'], y_pos[i],
                xerr=[[top_20.iloc[i]['prevalence'] - top_20.iloc[i]['ci_lower']],
                      [top_20.iloc[i]['ci_upper'] - top_20.iloc[i]['prevalence']]],
                fmt='none', color='black', capsize=3, linewidth=1)
    
    ax.text(top_20.iloc[i]['prevalence'] + 0.005, y_pos[i], 
            f"{top_20.iloc[i]['prevalence']*100:.1f}%",
            va='center', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(top_20['state_name'], fontsize=10)
ax.set_xlabel('Diabetes Prevalence', fontsize=12, fontweight='bold')
ax.set_title('Top 20 States by Diabetes Prevalence', fontsize=14, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))

plt.tight_layout()
plt.savefig('../outputs/figures/state_prevalence_top20.png', dpi=300, bbox_inches='tight')
print("\nSaved: state_prevalence_top20.png")
plt.close()

# Figure 2: Prevalence distribution histogram
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(state_df['prevalence'], bins=15, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(mean_prevalence, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_prevalence:.1%}')
ax.axvline(high_burden_threshold, color='orange', linestyle='--', linewidth=2, 
           label=f'High-Burden Threshold: {high_burden_threshold:.1%}')

ax.set_xlabel('Diabetes Prevalence', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of States', fontsize=12, fontweight='bold')
ax.set_title('Distribution of State-Level Diabetes Prevalence', fontsize=14, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/figures/prevalence_distribution.png', dpi=300, bbox_inches='tight')
print("Saved: prevalence_distribution.png")
plt.close()

# Figure 3: State ranking visualization
fig, ax = plt.subplots(figsize=(14, 10))

all_states_sorted = state_df.sort_values('prevalence', ascending=True)
y_pos = np.arange(len(all_states_sorted))

colors_map = []
for prev in all_states_sorted['prevalence']:
    if prev >= high_burden_threshold:
        colors_map.append('red')
    elif prev >= mean_prevalence:
        colors_map.append('orange')
    else:
        colors_map.append('green')

ax.barh(y_pos, all_states_sorted['prevalence'].values, color=colors_map, alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(all_states_sorted['state_name'], fontsize=8)
ax.set_xlabel('Diabetes Prevalence', fontsize=12, fontweight='bold')
ax.set_title('State Diabetes Prevalence Rankings', fontsize=14, fontweight='bold', pad=20)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
ax.axvline(mean_prevalence, color='black', linestyle='--', linewidth=1, alpha=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.7, label='High Burden (Top 25%)'),
    Patch(facecolor='orange', alpha=0.7, label='Above Average'),
    Patch(facecolor='green', alpha=0.7, label='Below Average')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/figures/state_rankings.png', dpi=300, bbox_inches='tight')
print("Saved: state_rankings.png")
plt.close()

# Save state-level data table
state_export = state_df[['state_name', 'prevalence', 'ci_lower', 'ci_upper', 'n', 'diabetes_cases']].copy()
state_export.columns = ['State', 'Prevalence', 'CI_Lower', 'CI_Upper', 'Sample_Size', 'Diabetes_Cases']
state_export.to_csv('../outputs/tables/state_level_prevalence.csv', index=False)
print("Saved: state_level_prevalence.csv")

# Create intervention priority matrix
# Priority based on: high prevalence + large population
state_df['priority_score'] = state_df['prevalence'] * np.log(state_df['n'])
state_df_priority = state_df.sort_values('priority_score', ascending=False)

intervention_priority = state_df_priority.head(10)[['state_name', 'prevalence', 'n', 'priority_score']].copy()
intervention_priority.columns = ['State', 'Prevalence', 'Population_Sample', 'Priority_Score']
intervention_priority.to_csv('../outputs/tables/intervention_priority_states.csv', index=False)
print("Saved: intervention_priority_states.csv")

print("\nTop 10 Intervention Priority States:")
print("="*70)
for _, row in intervention_priority.iterrows():
    print(f"{row['State']:<20} Prev: {row['Prevalence']:>6.1%}  Sample: {row['Population_Sample']:>6,}  Score: {row['Priority_Score']:>6.1f}")
print("="*70)

# Summary statistics for documentation
geographic_summary = {
    'total_states_analyzed': int(len(state_df)),
    'mean_prevalence': float(mean_prevalence),
    'std_prevalence': float(std_prevalence),
    'min_prevalence': float(state_df['prevalence'].min()),
    'max_prevalence': float(state_df['prevalence'].max()),
    'prevalence_range': float(prevalence_range),
    'highest_state': state_df.iloc[0]['state_name'],
    'lowest_state': state_df.iloc[-1]['state_name'],
    'high_burden_threshold': float(high_burden_threshold),
    'high_burden_states_count': int(len(high_burden_states)),
    'high_burden_states': high_burden_states['state_name'].tolist()
}

with open('../outputs/tables/geographic_summary.json', 'w') as f:
    json.dump(geographic_summary, f, indent=2)

print("Saved: geographic_summary.json")

# Final summary
print("\n" + "="*70)
print("NOTEBOOK 6 COMPLETE: GEOGRAPHIC MAPPING")
print("="*70)
print(f"States analyzed: {len(state_df)}")
print(f"Geographic disparity: {prevalence_range:.2%} range")
print(f"  Highest: {state_df.iloc[0]['state_name']} ({state_df.iloc[0]['prevalence']:.1%})")
print(f"  Lowest: {state_df.iloc[-1]['state_name']} ({state_df.iloc[-1]['prevalence']:.1%})")
print(f"\nHigh-burden states: {len(high_burden_states)} (>{high_burden_threshold:.1%})")
print(f"Intervention priority regions identified")
print(f"\nFigures created: 3")
print(f"  - state_prevalence_top20.png")
print(f"  - prevalence_distribution.png")
print(f"  - state_rankings.png")
print(f"\nTables created: 3")
print(f"  - state_level_prevalence.csv")
print(f"  - intervention_priority_states.csv")
print(f"  - geographic_summary.json")
print("="*70)
print("\n" + "="*70)
print("ALL 6 NOTEBOOKS COMPLETE - PROJECT FINISHED")
print("="*70)

Environment loaded successfully
Loaded: 10,000 respondents
States represented: 54

Weighted prevalence function loaded

Calculating state-level diabetes prevalence...
States with sufficient data: 54

Top 10 States by Diabetes Prevalence:
Georgia               19.6%  [14.8%, 24.4%]  n=152
Kentucky              19.0%  [14.8%, 23.1%]  n=199
Utah                  17.3%  [13.0%, 21.6%]  n=167
nan                   16.6%  [12.4%, 20.9%]  n=173
California            16.3%  [12.1%, 20.6%]  n=175
Rhode Island          16.3%  [12.3%, 20.3%]  n=183
Oklahoma              15.5%  [11.7%, 19.3%]  n=198
Massachusetts         15.4%  [11.3%, 19.5%]  n=171
Maine                 15.3%  [11.3%, 19.3%]  n=182
Wisconsin             14.9%  [10.9%, 18.9%]  n=175

Bottom 10 States by Diabetes Prevalence:
South Dakota          10.5%  [7.3%, 13.7%]  n=206
Iowa                  10.4%  [7.0%, 13.8%]  n=174
Louisiana             10.4%  [6.9%, 13.8%]  n=166
Indiana                9.8%  [6.5%, 13.1%]  n=181
nan       